# Predicting UK used-car prices
**Can we estimate a listing price from the car's specifications?** We compare three regression models on 97,711 UK listings. In the verified run, Random Forest averaged £1,138 absolute error on unseen profiles. The estimate applies to this historical UK sample, covering model years through 2020.

Run all cells in order. The training section prints validation errors after every 10 trees. Edit the car in the final cell to make your own estimate.

[Dataset and cleaning code at the project revision](https://github.com/MazenBassem/Used-Car-Price-Prediction-Market-Pattern-Discovery-Data-Mining/tree/afa7ea823febad0e4ebcdf0cf61a9b9cf1c1fb8e)

In [ ]:
from pathlib import Path
from datetime import datetime
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from IPython.display import display
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor

# Jupyter can start the kernel from the repository or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.uk_modeling import load_splits, pipeline_for

output_dir = project_root / "outputs" / "uk-modeling"
output_dir.mkdir(parents=True, exist_ok=True)


def progress(message):
    line = f"{datetime.now():%H:%M:%S}  {message}"
    print(line, flush=True)
    with (output_dir / "baseline_training.log").open("a", encoding="utf-8") as log:
        print(line, file=log)

plt.rcParams.update({"figure.figsize": (8, 3.8), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})


def metrics(actual, predicted):
    return dict(MAE=mean_absolute_error(actual, predicted),
                RMSE=root_mean_squared_error(actual, predicted), R2=r2_score(actual, predicted))

train_set, validation, test = load_splits()
development = pd.concat([train_set, validation]).sort_index()
progress(f"Loaded {len(development) + len(test):,} listings | training {len(train_set):,} | validation {len(validation):,} | test {len(test):,}")
display(train_set.head(4))

## Explore the training listings
Identical vehicle profiles stay in the same partition. We use the training rows for exploration, validation rows to compare models, and the reserved test rows for the final evaluation. Each bar below is the median listing price for one brand, including its mix of models and ages.

In [ ]:
medians = train_set.groupby("brand").price.median().sort_values()
ax = medians.plot.barh(color="#27766e", title="Median listing price by brand")
ax.set(xlabel="Median price (£)", ylabel="")
ax.xaxis.set_major_formatter(StrMethodFormatter("£{x:,.0f}"))
plt.show()

## Establish comparison models
Linear regression uses one-hot categories. The trees use integer-encoded categories. Each pipeline learns missing-value replacements from its own fitting rows. MAE measures average absolute error in pounds. RMSE gives more weight to large misses.

In [ ]:
models, scores = {}, {}
for name, regressor in [("Linear regression", LinearRegression()),
                        ("Decision tree", DecisionTreeRegressor(min_samples_leaf=4, random_state=42))]:
    progress(f"Fitting {name}")
    model = pipeline_for(regressor).fit(train_set, train_set.price)
    models[name] = model
    scores[name] = metrics(validation.price, model.predict(validation))
    progress(f"{name} | validation MAE £{scores[name]['MAE']:,.0f} | RMSE £{scores[name]['RMSE']:,.0f}")

## Grow the Random Forest
A forest grows by adding independently fitted trees. Each checkpoint keeps the existing trees, adds ten, then scores the entire forest on the same validation set. We reserve the test set until model selection is complete.

In [ ]:
forest = RandomForestRegressor(min_samples_leaf=2, max_features=0.85, random_state=42,
                               warm_start=True, n_jobs=4)
model = pipeline_for(forest)
prep = model.steps[0][1]
x_train = prep.fit_transform(train_set)
x_valid = prep.transform(validation)
history = []
for trees in range(10, 121, 10):
    forest.set_params(n_estimators=trees).fit(x_train, train_set.price)
    score = metrics(validation.price, forest.predict(x_valid))
    history.append({"trees": trees, **score})
    progress(f"Checkpoint {trees//10:02d}/12 | {trees:3d} trees | validation MAE £{score['MAE']:,.2f} | RMSE £{score['RMSE']:,.2f}")
models["Random forest"] = model
scores["Random forest"] = score
curve = pd.DataFrame(history)
ax = curve.plot(x="trees", y="MAE", marker="o", legend=False, color="#27766e",
                title="Validation error as the forest grows")
ax.set(xlabel="Trees in forest", ylabel="Mean absolute error (£)", xticks=curve.trees)
plt.show()

## Select by validation MAE and test once
The comparison table uses the same validation rows for all three candidates. We refit the winner on training and validation combined. The final comparison includes a simple benchmark that always predicts the development-set median price.

In [ ]:
comparison = pd.DataFrame(scores).T.sort_values("MAE")
display(comparison.round(3))
winner = comparison.index[0]
progress(f"Refitting {winner} on {len(development):,} listings")
final = clone(models[winner])
if winner == "Random forest":
    final_prep, final_forest = (step for _, step in final.steps)
    x_dev = final_prep.fit_transform(development)
    for trees in range(10, 121, 10):
        final_forest.set_params(n_estimators=trees).fit(x_dev, development.price)
        progress(f"Final refit | {trees:3d}/120 trees")
else:
    final.fit(development, development.price)
predictions = final.predict(test)
test_score = metrics(test.price, predictions)
benchmark = metrics(test.price, np.full(len(test), development.price.median()))
display(pd.DataFrame({winner: test_score, "Median baseline": benchmark}).T.round(3))
progress(f"Held-out test | MAE £{test_score['MAE']:,.2f} | RMSE £{test_score['RMSE']:,.2f} | R² {test_score['R2']:.3f}")
model_path = output_dir / "uk_price.joblib"
temporary_path = model_path.with_suffix(".tmp")
joblib.dump(final, temporary_path, compress=3)
temporary_path.replace(model_path)
progress(f"Saved {model_path.name}")

In [ ]:
fig, ax = plt.subplots()
density = ax.hexbin(test.price, predictions, gridsize=45, bins="log", mincnt=1,
                    xscale="log", yscale="log", cmap="viridis")
limit = [min(test.price.min(), predictions.min()), max(test.price.max(), predictions.max())]
ax.plot(limit, limit, "--", color="#aa342c", label="Exact prediction")
ax.set(xlabel="Listed price (£, log scale)", ylabel="Predicted price (£, log scale)",
       title="How closely do predictions track held-out prices?")
ax.legend()
fig.colorbar(density, ax=ax, label="Listings per bin")
plt.show()

## Price a car
Change the example below and run this cell again. Use miles for mileage, litres for engine size, pounds for annual road tax, and UK miles per gallon for fuel economy. The diagonal in the chart above marks an exact estimate. Cars farther from the diagonal had larger proportional errors.

In [ ]:
car = {"brand": "Ford", "model": "Fiesta", "year": 2018, "mileage": 35000,
       "transmission": "Manual", "fuel_type": "Petrol", "engine_size": 1.0,
       "tax": 150, "mpg": 60.1}
saved_model = joblib.load(model_path)
estimate = saved_model.predict(pd.DataFrame([car]))[0]
progress(f"{car['year']} {car['brand']} {car['model']} | estimated UK listing price £{estimate:,.0f}")